# 🛡️ AE-WGAN NSL-KDD — Hypothèse dataset équilibré (70,343 samples)

**Paper page 10:** *"the total number of samples used was 70,343, which balanced the number of samples in each attack category"*

### Hypothèse : le paper downsample Normal et DoS
- 70,343 / 5 catégories ≈ **14,069 par catégorie**
- Normal : 67,343 → 14,069 (downsampling)
- DoS : 45,927 → 14,069 (downsampling)
- Probe : 11,656 → 14,069 (AE-WGAN +2,413)
- R2L : 995 → 14,069 (AE-WGAN +13,074)
- U2R : 52 → 14,069 (AE-WGAN +14,017)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow numpy pandas scikit-learn matplotlib scipy

In [ ]:
import os, urllib.request, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten, Concatenate)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
for f,u in {"KDDTrain+.txt":url+"KDDTrain%2B.txt","KDDTest+.txt":url+"KDDTest%2B.txt"}.items():
    if not os.path.exists(f): urllib.request.urlretrieve(u, f)

COLS = ['duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level']

ATK = {'normal':'Normal','back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS',
    'smurf':'DoS','teardrop':'DoS','mailbomb':'DoS','apache2':'DoS',
    'processtable':'DoS','udpstorm':'DoS','satan':'Probe','ipsweep':'Probe',
    'nmap':'Probe','portsweep':'Probe','mscan':'Probe','saint':'Probe',
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L',
    'multihop':'R2L','warezmaster':'R2L','warezclient':'R2L','spy':'R2L',
    'xlock':'R2L','xsnoop':'R2L','snmpguess':'R2L','snmpgetattack':'R2L',
    'httptunnel':'R2L','sendmail':'R2L','named':'R2L','worm':'R2L',
    'buffer_overflow':'U2R','loadmodule':'U2R','rootkit':'U2R',
    'perl':'U2R','sqlattack':'U2R','xterm':'U2R','ps':'U2R'}

FEATS = ['protocol_type','flag','land','wrong_fragment','logged_in',
    'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_srv_rerror_rate','difficulty_level']

df_tr=pd.read_csv("KDDTrain+.txt",header=None,names=COLS)
df_te=pd.read_csv("KDDTest+.txt",header=None,names=COLS)
df_tr['label']=df_tr['label'].str.strip().str.rstrip('.')
df_te['label']=df_te['label'].str.strip().str.rstrip('.')

y_train_cat=df_tr['label'].map(ATK).fillna('Unknown')
y_test_cat=df_te['label'].map(ATK).fillna('Unknown')
y_train_detail=df_tr['label'].copy()

X_tr_df=df_tr[FEATS].copy(); X_te_df=df_te[FEATS].copy()
cat_cols=X_tr_df.select_dtypes(include=['object']).columns.tolist()
enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
X_tr_df[cat_cols]=enc.fit_transform(X_tr_df[cat_cols])
X_te_df[cat_cols]=enc.transform(X_te_df[cat_cols])
scaler=MinMaxScaler()
X_train_all=scaler.fit_transform(X_tr_df.values)
X_test_final=scaler.transform(X_te_df.values)
INPUT_DIM=24

# Target: 70,343 / 5 = 14,069 par catégorie
TOTAL_TARGET = 70343
N_CATEGORIES = 5
TARGET_PER_CAT = TOTAL_TARGET // N_CATEGORIES  # = 14,068

print(f"✅ Données chargées")
print(f"   Target total: {TOTAL_TARGET}, par catégorie: {TARGET_PER_CAT}")
print(f"\nDistribution originale:")
for cat in ['Normal','DoS','Probe','R2L','U2R']:
    n=(y_train_cat==cat).sum()
    action = 'DOWNSAMPLE' if n > TARGET_PER_CAT else 'AUGMENTER'
    delta = TARGET_PER_CAT - n
    print(f"  {cat:>8}: {n:>6} → {TARGET_PER_CAT} ({action} {abs(delta):+d})")

## 2. AE-WGAN

In [ ]:
class AEWGAN_Original:
    def __init__(self):
        self.img_shape=(4,6,1);self.latent_dim=32
        self.flat_dim=int(np.prod(self.img_shape))
        opt=tf.keras.optimizers.Adamax(0.00009)
        self.discriminator=self._bd()
        self.discriminator.compile(loss=self._wl,optimizer=opt,metrics=['accuracy'])
        self.generator=self._bg()
        self.encoder=self._be()
        self.discriminator.trainable=False
        z=Input(shape=(32,));img_=self.generator(z)
        img=Input(shape=(4,6,1));z_=self.encoder(img)
        self.combined=Model([z,img],[self.discriminator([z,img_]),self.discriminator([z_,img])])
        self.combined.compile(loss=[self._wl,self._wl],optimizer=opt)
    def _wl(self,yt,yp): return K.mean(yt*yp)
    def _be(self):
        m=Sequential([Flatten(input_shape=(4,6,1)),Dense(20,activation='relu'),Dense(32)])
        i=Input(shape=(4,6,1));return Model(i,m(i),name='enc')
    def _bg(self):
        m=Sequential([Dense(20,input_dim=32,activation='relu'),Dense(40,activation='relu'),
            Dense(80,activation='relu'),Dense(120,activation='relu'),Dense(160,activation='relu'),
            Dense(self.flat_dim,activation='relu'),Reshape((4,6,1))])
        z=Input(shape=(32,));return Model(z,m(z),name='gen')
    def _bd(self):
        z=Input(shape=(32,));img=Input(shape=(4,6,1))
        d=Concatenate()([z,Flatten()(img)])
        d=Dense(80,activation='relu')(d);d=Dense(40,activation='relu')(d)
        return Model([z,img],Dense(1,activation='sigmoid')(d),name='disc')
    def train(self,X,epochs=101,batch_size=32,verbose=1):
        r=X.reshape(-1,4,6,1);n=r.shape[0]
        if n<batch_size:batch_size=max(n//2,2)
        v=np.ones((batch_size,1));f=np.zeros((batch_size,1))
        for e in range(epochs):
            z=np.random.normal(0,1,(batch_size,32))
            gi=self.generator.predict(z,verbose=0)
            idx=np.random.randint(0,n,batch_size);imgs=r[idx]
            ze=self.encoder.predict(imgs,verbose=0)
            self.discriminator.train_on_batch([ze,imgs],v)
            self.discriminator.train_on_batch([z,gi],f)
            self.combined.train_on_batch([z,imgs],[v,f])
            if verbose and (e+1)%max(1,epochs//5)==0:
                print(f"      Epoch {e+1}/{epochs}")
    def generate_batch(self,n):
        return self.generator.predict(np.random.normal(0,1,(n,32)),verbose=0,batch_size=256).reshape(-1,self.flat_dim)

print("✅ AE-WGAN défini")

## 3. Construction du dataset équilibré (70,343 samples)

In [ ]:
print("=" * 60)
print(f"CONSTRUCTION DATASET ÉQUILIBRÉ ({TOTAL_TARGET} samples)")
print("=" * 60)

balanced_X = []
balanced_y = []
start = time.time()

for category in ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']:
    mask = y_train_cat == category
    X_cat = X_train_all[mask]
    n_orig = X_cat.shape[0]

    if n_orig >= TARGET_PER_CAT:
        # DOWNSAMPLE: prendre un sous-échantillon aléatoire
        idx = np.random.choice(n_orig, TARGET_PER_CAT, replace=False)
        balanced_X.append(X_cat[idx])
        balanced_y.extend([category] * TARGET_PER_CAT)
        print(f"  [{category}] ⬇️ Downsample {n_orig} → {TARGET_PER_CAT}")

    elif n_orig >= 10:
        # AUGMENTER: garder l'original + générer synthétique
        n_to_gen = TARGET_PER_CAT - n_orig
        epochs = 300 if n_orig < 2000 else 101

        print(f"\n  [{category}] 🧠 Training ({n_orig} samples, +{n_to_gen}, {epochs} epochs)...")
        model = AEWGAN_Original()
        bs = min(32, max(n_orig // 2, 2))
        model.train(X_cat, epochs=epochs, batch_size=bs, verbose=1)

        synthetic = model.generate_batch(n_to_gen)
        combined = np.vstack([X_cat, synthetic])
        balanced_X.append(combined)
        balanced_y.extend([category] * TARGET_PER_CAT)
        print(f"  [{category}] ✅ {n_orig} + {n_to_gen} = {TARGET_PER_CAT}")

    else:
        print(f"  [{category}] ⏭️ {n_orig} < 10 — skip")

elapsed = (time.time() - start) / 60
print(f"\n--- {elapsed:.1f} min ---")

# Assembler le dataset équilibré
X_balanced = np.vstack(balanced_X)
y_balanced_cat = pd.Series(balanced_y)

print(f"\n{'='*60}")
print(f"  Dataset équilibré: {X_balanced.shape[0]} samples")
print(f"{'='*60}")
print(y_balanced_cat.value_counts())

## 4. Classifieurs + Évaluation par catégorie

In [ ]:
def build_cnn(dim,nc):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential([layers.Input(shape=(dim,1)),layers.Conv1D(40,3,activation='relu',padding='same'),
        layers.Dropout(0.1),layers.Flatten(),layers.Dense(40,activation='relu'),
        layers.Dense(out,activation=act)],name='CNN')
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def build_gru(dim,nc):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='GRU'); m.add(layers.Input(shape=(dim,1)))
    if nc==2:
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10,activation='relu'));m.add(layers.Dense(10,activation='relu'))
    else:
        m.add(layers.GRU(30,activation='relu',return_sequences=True));m.add(layers.Dropout(0.1))
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def eval_per_category(name, clf, X_tr, y_tr_cat, X_te, y_te_cat, dim):
    le = LabelEncoder()
    all_cls = sorted(list(set(y_tr_cat.unique()) | set(y_te_cat.unique())))
    le.fit(all_cls)
    y_tr=le.transform(y_tr_cat); y_te=le.transform(y_te_cat); nc=len(all_cls)
    if name in ['CNN','GRU']:
        X3=X_tr.reshape(-1,dim,1); Xt3=X_te.reshape(-1,dim,1)
        es=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
        clf.fit(X3,y_tr,epochs=50,batch_size=256,validation_split=0.1,callbacks=[es],verbose=0)
        yp=clf.predict(Xt3,verbose=0); y_pred=np.argmax(yp,axis=1)
    else:
        clf.fit(X_tr,y_tr); y_pred=clf.predict(X_te)
    r={'classifier':name,'accuracy':accuracy_score(y_te,y_pred),
       'f1_weighted':f1_score(y_te,y_pred,average='weighted',zero_division=0)}
    for cat in ['Normal','DoS','Probe','R2L','U2R']:
        if cat in le.classes_:
            ci=le.transform([cat])[0]
            ytb=(y_te==ci).astype(int);ypb=(y_pred==ci).astype(int)
            r[f'f1_{cat}']=f1_score(ytb,ypb,zero_division=0)
            r[f'prec_{cat}']=precision_score(ytb,ypb,zero_division=0)
            r[f'rec_{cat}']=recall_score(ytb,ypb,zero_division=0)
        else:
            r[f'f1_{cat}']=0.0;r[f'prec_{cat}']=0.0;r[f'rec_{cat}']=0.0
    return r

print("✅ Prêt")

In [ ]:
clf_builders = {
    'DT': lambda nc: DecisionTreeClassifier(max_depth=9, random_state=42),
    'RF': lambda nc: RandomForestClassifier(100, max_depth=10, random_state=42, n_jobs=-1),
    'CNN': lambda nc: build_cnn(INPUT_DIM, nc),
    'GRU': lambda nc: build_gru(INPUT_DIM, nc),
}
all_cls = sorted(list(set(y_train_cat.unique()) | set(y_test_cat.unique())))
nc = len(all_cls)

# --- BASELINE (dataset original complet) ---
print("\n" + "="*70)
print(f"  BASELINE (original, {X_train_all.shape[0]} samples)")
print("="*70)
base_res = []
for name, builder in clf_builders.items():
    print(f"  {name}...", end=" ")
    r = eval_per_category(name, builder(nc), X_train_all, y_train_cat, X_test_final, y_test_cat, INPUT_DIM)
    base_res.append(r)
    print(f"R2L={r['f1_R2L']:.3f} U2R={r['f1_U2R']:.3f} Global={r['f1_weighted']:.3f}")

# --- DATASET ÉQUILIBRÉ (70,343 samples) ---
print("\n" + "="*70)
print(f"  AE-WGAN ÉQUILIBRÉ ({X_balanced.shape[0]} samples)")
print("="*70)
bal_res = []
for name, builder in clf_builders.items():
    print(f"  {name}...", end=" ")
    r = eval_per_category(name, builder(nc), X_balanced, y_balanced_cat, X_test_final, y_test_cat, INPUT_DIM)
    bal_res.append(r)
    print(f"R2L={r['f1_R2L']:.3f} U2R={r['f1_U2R']:.3f} Global={r['f1_weighted']:.3f}")

## 5. Comparaison détaillée

In [ ]:
cats = ['Normal','DoS','Probe','R2L','U2R']

print("\n" + "="*80)
print("  COMPARAISON: Original vs Équilibré (70K) — F1 par catégorie")
print("="*80)
print(f"\n  {'Clf':<6} {'Type':<14} {'Normal':>8} {'DoS':>8} {'Probe':>8} {'R2L':>8} {'U2R':>8} {'Global':>8}")
print(f"  {'─'*68}")

for b, e in zip(base_res, bal_res):
    name = b['classifier']
    print(f"  {name:<6} {'Baseline':<14}", end="")
    for c in cats: print(f" {b[f'f1_{c}']:>7.3f}", end="")
    print(f" {b['f1_weighted']:>7.3f}")
    
    print(f"  {'':6} {'Équilibré 70K':<14}", end="")
    for c in cats: print(f" {e[f'f1_{c}']:>7.3f}", end="")
    print(f" {e['f1_weighted']:>7.3f}")
    
    print(f"  {'':6} {'Δ':<14}", end="")
    for c in cats:
        d = e[f'f1_{c}'] - b[f'f1_{c}']
        icon = '✅' if d > 0.01 else '❌' if d < -0.01 else '➖'
        print(f" {'+' if d>=0 else ''}{d:>5.3f}{icon}", end="")
    dg = e['f1_weighted'] - b['f1_weighted']
    print(f" {'+' if dg>=0 else ''}{dg:>6.3f}")
    print()

# Paper comparison
print("\n" + "="*60)
print("  COMPARAISON AVEC LE PAPER (Table 6, CNN)")
print("="*60)
print(f"  {'':15} {'Normal':>8} {'DoS':>8} {'Probe':>8} {'R2L':>8} {'U2R':>8}")
print(f"  {'─'*55}")
print(f"  {'Paper AE-WGAN':<15}    0.890    0.890    0.680    0.650    0.000")
for r in bal_res:
    if r['classifier'] == 'CNN':
        print(f"  {'Notre Équilibré':<15}", end="")
        for c in cats: print(f" {r[f'f1_{c}']:>7.3f}", end="")
        print()
        print(f"  {'Écart':<15}", end="")
        paper_vals = {'Normal':0.89,'DoS':0.89,'Probe':0.68,'R2L':0.65,'U2R':0.0}
        for c in cats:
            d = r[f'f1_{c}'] - paper_vals[c]
            print(f" {'+' if d>=0 else ''}{d:>6.3f}", end="")
        print()

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('NSL-KDD: Baseline vs Équilibré 70K — F1 par catégorie', fontsize=16, fontweight='bold')
for idx, (b, e) in enumerate(zip(base_res, bal_res)):
    ax = axes[idx//2][idx%2]
    x = np.arange(len(cats)); w = 0.35
    vb = [b[f'f1_{c}'] for c in cats]; ve = [e[f'f1_{c}'] for c in cats]
    ax.bar(x-w/2, vb, w, label='Baseline', color='#3498db', alpha=0.8)
    ax.bar(x+w/2, ve, w, label='Équilibré 70K', color='#e74c3c', alpha=0.8)
    for i,(vbi,vei) in enumerate(zip(vb,ve)):
        d=vei-vbi
        if abs(d)>0.001:
            ax.annotate(f'{d:+.1%}',xy=(i+w/2,max(vei,0.01)+0.02),ha='center',fontsize=9,
                        color='green' if d>0 else 'red',fontweight='bold')
    ax.set_xlabel('Catégorie');ax.set_ylabel('F1 Score');ax.set_title(b['classifier'])
    ax.set_xticks(x);ax.set_xticklabels(cats);ax.legend();ax.set_ylim(0,1.15);ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('results_balanced_70K.png', dpi=150, bbox_inches='tight')
plt.show()

# CSV + Drive
rows = []
for r in base_res:
    for c in cats:
        rows.append({'classifier':r['classifier'],'type':'Baseline','category':c,
                     'f1':r[f'f1_{c}'],'precision':r[f'prec_{c}'],'recall':r[f'rec_{c}']})
for r in bal_res:
    for c in cats:
        rows.append({'classifier':r['classifier'],'type':'Balanced-70K','category':c,
                     'f1':r[f'f1_{c}'],'precision':r[f'prec_{c}'],'recall':r[f'rec_{c}']})
df = pd.DataFrame(rows)
df.to_csv('results_balanced_70K.csv', index=False)
import shutil
shutil.copy('results_balanced_70K.csv', '/content/drive/MyDrive/')
shutil.copy('results_balanced_70K.png', '/content/drive/MyDrive/')
print("\n✅ Sauvegardé et copié sur Drive")
print(df.to_string(index=False))